# BIO-NN Experiment 2: Compare All Neuron Models

Benchmark all 7 neuron types on speed, spike rate, and membrane dynamics.
Tests: LIF, Adaptive LIF, Izhikevich, Dual LIF, AdEx, Resonate-and-Fire, SpikingBrain.

### 2.1 Setup

In [ ]:
import torch
import time
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from bio_nn.neurons import create_neuron

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

NEURON_MODELS = ["lif", "adaptive_lif", "izhikevich", "dual_lif", "adex", "resonate_fire", "spiking_brain"]
print(f"Testing {len(NEURON_MODELS)} neuron models: {NEURON_MODELS}")

### 2.2 Benchmark Each Neuron Type

In [ ]:
results = []
all_data = {}

for name in NEURON_MODELS:
    try:
        neuron = create_neuron(name, 256).to(device)
        state = neuron._get_initial_state(32, device)
        x = torch.randn(32, 256).to(device)

        # Warmup
        for _ in range(10):
            spikes, mem, state = neuron(x, state)

        # Benchmark
        spike_log = []
        mem_log = []
        start = time.time()
        for _ in range(200):
            spikes, mem, state = neuron(x, state)
            spike_log.append(spikes.cpu())
            mem_log.append(mem.cpu())
        elapsed = time.time() - start

        spike_tensor = torch.stack(spike_log)
        mem_tensor = torch.stack(mem_log)

        results.append({
            "Model": name,
            "Spike Rate": spike_tensor.float().mean().item(),
            "Time (s)": elapsed,
            "Steps/sec": 200 / elapsed,
            "Membrane (mean)": mem_tensor.mean().item(),
            "Membrane (std)": mem_tensor.std().item(),
            "Membrane (range)": mem_tensor.max().item() - mem_tensor.min().item()
        })

        all_data[name] = {
            "spikes": spike_tensor,
            "membrane": mem_tensor
        }
        print(f"  {name:20s} - spike_rate={spike_tensor.float().mean().item():.4f}, time={elapsed:.3f}s")

    except Exception as e:
        results.append({"Model": name, "Error": str(e)[:80]})
        print(f"  {name:20s} - FAILED: {e}")

df = pd.DataFrame(results)
print("\n" + "=" * 70)
print(df.to_string(index=False))

### 2.3 Speed Comparison Chart

In [ ]:
valid = [r for r in results if "Error" not in r]
models = [r["Model"] for r in valid]
times = [r["Time (s)"] for r in valid]
rates = [r["Spike Rate"] for r in valid]

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Speed
colors = plt.cm.viridis(np.linspace(0.2, 0.8, len(models)))
bars = axes[0].barh(models, times, color=colors)
axes[0].set_xlabel("Time (s) for 200 steps")
axes[0].set_title("Computation Speed", fontweight='bold')
for bar, t in zip(bars, times):
    axes[0].text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2,
                 f'{t:.3f}s', va='center', fontsize=9)

# Spike Rate
bars = axes[1].barh(models, rates, color=colors)
axes[1].set_xlabel("Mean Spike Rate")
axes[1].set_title("Spike Rate", fontweight='bold')
for bar, r in zip(bars, rates):
    axes[1].text(bar.get_width() + 0.001, bar.get_y() + bar.get_height()/2,
                 f'{r:.4f}', va='center', fontsize=9)

# Efficiency (rate / time)
efficiency = [r / t for r, t in zip(rates, times)]
bars = axes[2].barh(models, efficiency, color=colors)
axes[2].set_xlabel("Spike Rate / Second")
axes[2].set_title("Efficiency", fontweight='bold')

plt.suptitle("Neuron Model Benchmark", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('neuron_benchmark.png', dpi=150, bbox_inches='tight')
plt.show()

### 2.4 Spike Raster Comparison

In [ ]:
n_models = len(all_data)
fig, axes = plt.subplots(n_models, 2, figsize=(14, 3 * n_models))

for i, (name, data) in enumerate(all_data.items()):
    # Spike raster (first 100 steps, first 128 neurons)
    spike_matrix = data["spikes"][:100, 0, :128].numpy().T
    axes[i, 0].imshow(spike_matrix, aspect='auto', cmap='hot', interpolation='nearest')
    axes[i, 0].set_ylabel(name, fontweight='bold', fontsize=10)
    axes[i, 0].set_xticks([])
    if i == 0:
        axes[i, 0].set_title("Spike Raster", fontweight='bold')

    # Membrane potential (first 4 neurons)
    for n in range(min(4, data["membrane"].shape[2])):
        axes[i, 1].plot(data["membrane"][:100, 0, n].numpy(), linewidth=0.8, label=f'n{n}')
    axes[i, 1].set_xlim([0, 100])
    axes[i, 1].legend(loc='upper right', fontsize=7, ncol=2)
    if i == 0:
        axes[i, 1].set_title("Membrane Potential", fontweight='bold')
    axes[i, 1].set_xlabel("Time Step" if i == n_models - 1 else "")

plt.suptitle("Neuron Dynamics Comparison", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('neuron_dynamics.png', dpi=150, bbox_inches='tight')
plt.show()

### 2.5 Membrane Potential Distribution

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

for i, (name, data) in enumerate(all_data.items()):
    mem_flat = data["membrane"].numpy().flatten()
    axes[i].hist(mem_flat, bins=50, color='steelblue', edgecolor='black', alpha=0.7)
    axes[i].set_title(name, fontweight='bold')
    axes[i].set_xlabel("Membrane Potential")
    axes[i].set_ylabel("Count")
    axes[i].axvline(x=np.mean(mem_flat), color='red', linestyle='--', label=f'mean={np.mean(mem_flat):.3f}')
    axes[i].legend(fontsize=8)

# Hide last subplot if odd number of models
if len(all_data) < len(axes):
    for j in range(len(all_data), len(axes)):
        axes[j].set_visible(False)

plt.suptitle("Membrane Potential Distributions", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('membrane_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

### 2.6 Download All Results

In [ ]:
from google.colab import files
for f in ['neuron_benchmark.png', 'neuron_dynamics.png', 'membrane_distributions.png']:
    try:
        files.download(f)
    except:
        print(f"{f} not found")